In [1]:
import pandas as pd
import numpy as np
import torch
import os, pickle, collections, json
import random
import warnings
warnings.filterwarnings("ignore") # avoid torchmetrics warning

import jax
import jax.numpy as jnp

from disk.model.disk import DiSK
from disk.dataset.wikidata_names import WikidataNames
from disk.data.stats import TextStats
from disk.data.mapper import map_frame
from disk.data.record import RecordLoader

from diffusion_linking.smc_clustering import SMCClusterer, BigramCluster, BigramMixture, resample_greedy, resample_stratified
from diffusion_linking.utils import DFWrapper

In [2]:

def estimate_probs(
    df: pd.DataFrame,
    model: DiSK,
    num_samples: int,
    device: str = "cpu",
    batch_size: int = 200,
    num_workers: int = 0,
    verbose: bool = False,
) -> np.ndarray:
    '''
    Modification of disk.linking.score.estimate_linking_scores to return the log probs for each cluster
    '''

    df.reset_index(inplace=True)

    # map to RecordData
    record_data = map_frame(df, model.schema, model.stats)

    # Build data loader if necessary
    if record_data.num_records <= batch_size:
        loader = [record_data]
    else:
        loader = RecordLoader(
            data=record_data,
            input_nodes=np.arange(record_data.num_records),
            batch_size=batch_size,
            num_workers=num_workers,
            shuffle=False,
        )

    model.to(device)
    disk_model = model.model

    # get all log probs in one array
    all_log_probs = []
    with torch.no_grad():
        for data in loader:
            data.to(device)
            log_probs = disk_model.monte_carlo_log_probs(data, num_samples=num_samples)
            log_probs = log_probs.detach().cpu().numpy()
            all_log_probs.append(log_probs)
    log_probs = np.concatenate(all_log_probs, axis=0)

    return log_probs

def score_fn(rng, cluster_data, num_samples = 10, batch_size = 1):
    df = pd.DataFrame.from_records([{'names': names} for names in cluster_data])
    log_probs = estimate_probs(df, model, num_samples, device, batch_size=batch_size)    
    return log_probs    

In [3]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
schema = WikidataNames.schema
stats = {"names": TextStats("names", "t5-small", 32128)}

checkpoint = "checkpoints/DiSK_Autoregressive/checkpoints/best.ckpt"
model = DiSK.load_from_checkpoint(checkpoint, schema=schema, stats=stats)
model.to(device)
model.model.use_diffusion_weights = True

Load first part of the Wikinames dataset.

In [4]:
data = []
with open('data/candidates_val.jsonl') as file:
    for line in file:
        entry = json.loads(line)
        names = entry['entity1']['properties']['names'] + entry['entity2']['properties']['names']
        for name in names:
            if name not in data:
                data.append(name)
                
dataset = DFWrapper(pd.DataFrame.from_records([{'name': name} for name in [''] + data[:750]]))

Load bigram prior model pretrained on Wikipedia titles.

In [5]:
with open(f'data/wikipedia_names_ngram_counts.pickle', 'rb') as handle:
    count_dict = pickle.load(handle)
prior = collections.defaultdict(lambda: count_dict['<UNK>'], count_dict)

rng = jax.random.PRNGKey(1)

Splitting into clusters of similar names with the bigram model - this is a first attempt at splitting the clustering problem into smaller independent subproblems. This part is not a proper SMC sampler, as the assignment with the highest probability is chosen each time.

In [6]:
alpha = 1
surrogate = BigramMixture(alpha, 3, prior)
max_evals = 0; max_particles = 1
clusterer = SMCClusterer(dataset, lambda x, y: jnp.zeros((len(x),)), max_evals, max_particles, surrogate, resample_fn=resample_greedy, ClusterClass=BigramCluster)
clusterer.cluster(rng)
clusterer.summary(print_cluster_data = True)

100%|████████████████████████████| 749/749 [03:12<00:00,  3.89it/s, Particles=1]



Particle 0, weight 1, 293 clusters, 750 points, [12, 8, 8, 8, 8, 8, 7, 7, 7, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
['William Moyer', 'Walter Camp', 'Walter Chauncey Camp', 'Sonny Boy Wi

Using DiSK to cluster these subsets. The Dirichlet process concentration is raised to 10 to discourage merges without strong evidence, since most names have a very small number of variants. The subproblems are quite small, so the max_evals parameter is set high enough that the surrogate model is not used.

In [7]:
surrogate2 = BigramMixture(10, .01, prior)

In [8]:
cl = 0
dataset2 = DFWrapper(pd.DataFrame.from_records([{'name': name} for name in [''] + clusterer.state.retrieve_cluster_data(sorted(clusterer.state.particles[0], key=lambda c: clusterer.state.clusters[c].size, reverse=True)[cl]) ]))

Using only one particle and taking the assignment with the highest score each time ("greedy" resampling strategy): 

In [9]:
torch.manual_seed(1)
max_evals = 300; max_particles = 1
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_greedy, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

100%|██████████████████████████████| 11/11 [00:07<00:00,  1.54it/s, Particles=1]


Particle 0, weight 1, 9 clusters, 12 points, [3, 2, 1, 1, 1, 1, 1, 1, 1]
['Walter James', 'William Thompson Walters', 'William Walters']
['Walter Camp', 'Walter Chauncey Camp']
['Sonny Boy Williamson']
['William Halsted']
['Donald Trump']
['James Donald Cameron']
['William Kingston']
['William Moyer']
['William Dickinson']



Using an SMC sampler:

In [10]:
torch.manual_seed(1)
max_evals = 300; max_particles = 10
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_stratified, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

100%|██████████████████████████████| 11/11 [00:24<00:00,  2.22s/it, Particles=3]


Particle 0, weight 0.4, 8 clusters, 12 points, [2, 2, 2, 2, 1, 1, 1, 1]
['Sonny Boy Williamson', 'William Dickinson']
['Walter Camp', 'Walter Chauncey Camp']
['William Thompson Walters', 'William Walters']
['William Moyer', 'William Kingston']
['Walter James']
['William Halsted']
['Donald Trump']
['James Donald Cameron']

Particle 1, weight 0.3, 8 clusters, 12 points, [3, 2, 2, 1, 1, 1, 1, 1]
['William Moyer', 'William Kingston', 'William Halsted']
['Walter Camp', 'Walter Chauncey Camp']
['William Thompson Walters', 'William Walters']
['Sonny Boy Williamson']
['Walter James']
['Donald Trump']
['James Donald Cameron']
['William Dickinson']

Particle 2, weight 0.3, 10 clusters, 12 points, [2, 2, 1, 1, 1, 1, 1, 1, 1, 1]
['Walter Camp', 'Walter Chauncey Camp']
['William Thompson Walters', 'William Walters']
['Sonny Boy Williamson']
['Walter James']
['William Halsted']
['Donald Trump']
['James Donald Cameron']
['William Kingston']
['William Moyer']
['William Dickinson']



The greedy sampler tends to create clusters that are too large, as the model struggles with telling very similar names apart. Ground truth clusters are typically recovered (with somewhat reasonable weights) using SMC sampling, as the true clustering is given higher weight by the model after more data is observed.

Clustering results follow for the first few of the other sub-problems. These clusterers could be run in parallel with the one that generates the sub-problems, with one step of a sub-clusterer being executed whenever a new datapoint is assigned to its sub-problem.

In [11]:
cl = 1
dataset2 = DFWrapper(pd.DataFrame.from_records([{'name': name} for name in [''] + clusterer.state.retrieve_cluster_data(sorted(clusterer.state.particles[0], key=lambda c: clusterer.state.clusters[c].size, reverse=True)[cl]) ]))

print('Greedy:')
torch.manual_seed(1)
max_evals = 300; max_particles = 1
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_greedy, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

print('SMC:')
torch.manual_seed(1)
max_evals = 300; max_particles = 10
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_stratified, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

Greedy:


100%|████████████████████████████████| 7/7 [00:03<00:00,  2.25it/s, Particles=1]



Particle 0, weight 1, 5 clusters, 8 points, [4, 1, 1, 1, 1]
['Charles Borromeo', 'Charles Hallows', 'Charles Houston', 'Charles Deane']
['Charles De Geer']
['Charles Dixon']
['Ernest Charles Drury']
['Sir Charles Fellows']

SMC:


100%|████████████████████████████████| 7/7 [00:09<00:00,  1.30s/it, Particles=6]


Particle 0, weight 0.3, 4 clusters, 8 points, [4, 2, 1, 1]
['Charles De Geer', 'Charles Borromeo', 'Charles Houston', 'Charles Deane']
['Charles Hallows', 'Sir Charles Fellows']
['Ernest Charles Drury']
['Charles Dixon']

Particle 1, weight 0.3, 5 clusters, 8 points, [3, 2, 1, 1, 1]
['Charles Borromeo', 'Charles Houston', 'Charles Deane']
['Charles Hallows', 'Sir Charles Fellows']
['Charles De Geer']
['Ernest Charles Drury']
['Charles Dixon']

Particle 2, weight 0.1, 3 clusters, 8 points, [5, 2, 1]
['Charles Borromeo', 'Charles Dixon', 'Charles Hallows', 'Charles Houston', 'Sir Charles Fellows']
['Charles De Geer', 'Charles Deane']
['Ernest Charles Drury']

Particle 3, weight 0.1, 4 clusters, 8 points, [5, 1, 1, 1]
['Charles Borromeo', 'Charles Dixon', 'Charles Hallows', 'Charles Houston', 'Charles Deane']
['Ernest Charles Drury']
['Charles De Geer']
['Sir Charles Fellows']

Particle 4, weight 0.1, 5 clusters, 8 points, [4, 1, 1, 1, 1]
['Charles Borromeo', 'Charles Dixon', 'Charles Ha

In [12]:
cl = 2
dataset2 = DFWrapper(pd.DataFrame.from_records([{'name': name} for name in [''] + clusterer.state.retrieve_cluster_data(sorted(clusterer.state.particles[0], key=lambda c: clusterer.state.clusters[c].size, reverse=True)[cl]) ]))

print('Greedy:')
torch.manual_seed(1)
max_evals = 300; max_particles = 1
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_greedy, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

print('SMC:')
torch.manual_seed(1)
max_evals = 300; max_particles = 10
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_stratified, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

Greedy:


100%|████████████████████████████████| 7/7 [00:02<00:00,  2.52it/s, Particles=1]



Particle 0, weight 1, 3 clusters, 8 points, [5, 2, 1]
['Edward Cargill', 'Edward Valentine', 'Edward Virginius Valentine', 'Edward Youde', 'Sir Edward Youde']
['Sir Edward Coke', 'Edward Coke']
['Edward Douglass White']

SMC:


100%|████████████████████████████████| 7/7 [00:07<00:00,  1.12s/it, Particles=3]


Particle 0, weight 0.4, 4 clusters, 8 points, [3, 2, 2, 1]
['Edward Cargill', 'Edward Youde', 'Sir Edward Youde']
['Sir Edward Coke', 'Edward Coke']
['Edward Valentine', 'Edward Virginius Valentine']
['Edward Douglass White']

Particle 1, weight 0.3, 4 clusters, 8 points, [2, 2, 2, 2]
['Edward Cargill', 'Edward Douglass White']
['Sir Edward Coke', 'Edward Coke']
['Edward Youde', 'Sir Edward Youde']
['Edward Valentine', 'Edward Virginius Valentine']

Particle 2, weight 0.3, 5 clusters, 8 points, [2, 2, 2, 1, 1]
['Edward Youde', 'Sir Edward Youde']
['Edward Valentine', 'Edward Virginius Valentine']
['Sir Edward Coke', 'Edward Coke']
['Edward Cargill']
['Edward Douglass White']



In [13]:
cl = 3
dataset2 = DFWrapper(pd.DataFrame.from_records([{'name': name} for name in [''] + clusterer.state.retrieve_cluster_data(sorted(clusterer.state.particles[0], key=lambda c: clusterer.state.clusters[c].size, reverse=True)[cl]) ]))

print('Greedy:')
torch.manual_seed(1)
max_evals = 300; max_particles = 1
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_greedy, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

print('SMC:')
torch.manual_seed(1)
max_evals = 300; max_particles = 10
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_stratified, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

Greedy:


100%|████████████████████████████████| 7/7 [00:02<00:00,  2.42it/s, Particles=1]



Particle 0, weight 1, 3 clusters, 8 points, [5, 2, 1]
['Saint Indaletius', 'Saint Ambrose', 'St Ambrose', 'Ambrose', 'Ambrosius']
['Alec Rose', 'Sir Alec Rose']
['Alexey Rykov']

SMC:


100%|████████████████████████████████| 7/7 [00:06<00:00,  1.10it/s, Particles=1]


Particle 0, weight 1, 4 clusters, 8 points, [4, 2, 1, 1]
['Ambrosius', 'Saint Ambrose', 'St Ambrose', 'Ambrose']
['Alec Rose', 'Sir Alec Rose']
['Saint Indaletius']
['Alexey Rykov']



In [14]:
cl = 4
dataset2 = DFWrapper(pd.DataFrame.from_records([{'name': name} for name in [''] + clusterer.state.retrieve_cluster_data(sorted(clusterer.state.particles[0], key=lambda c: clusterer.state.clusters[c].size, reverse=True)[cl]) ]))

print('Greedy:')
torch.manual_seed(1)
max_evals = 300; max_particles = 1
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_greedy, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

print('SMC:')
torch.manual_seed(1)
max_evals = 300; max_particles = 10
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_stratified, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

Greedy:


100%|████████████████████████████████| 7/7 [00:02<00:00,  2.36it/s, Particles=1]



Particle 0, weight 1, 6 clusters, 8 points, [2, 2, 1, 1, 1, 1]
['SCO19', 'SO19']
['Walter Scott', 'Sir Walter Scott']
['Ted Sizer']
['CO19']
['Wannier']
['Walter Fred Bodmer']

SMC:


100%|████████████████████████████████| 7/7 [00:06<00:00,  1.06it/s, Particles=5]


Particle 0, weight 0.4, 4 clusters, 8 points, [4, 2, 1, 1]
['Ted Sizer', 'SCO19', 'CO19', 'SO19']
['Walter Scott', 'Sir Walter Scott']
['Walter Fred Bodmer']
['Wannier']

Particle 1, weight 0.3, 5 clusters, 8 points, [3, 2, 1, 1, 1]
['SCO19', 'CO19', 'SO19']
['Walter Scott', 'Sir Walter Scott']
['Wannier']
['Walter Fred Bodmer']
['Ted Sizer']

Particle 2, weight 0.1, 4 clusters, 8 points, [4, 2, 1, 1]
['Wannier', 'SCO19', 'CO19', 'SO19']
['Walter Scott', 'Sir Walter Scott']
['Walter Fred Bodmer']
['Ted Sizer']

Particle 3, weight 0.1, 5 clusters, 8 points, [2, 2, 2, 1, 1]
['SCO19', 'SO19']
['Walter Scott', 'Sir Walter Scott']
['Ted Sizer', 'CO19']
['Wannier']
['Walter Fred Bodmer']

Particle 4, weight 0.1, 6 clusters, 8 points, [2, 2, 1, 1, 1, 1]
['CO19', 'SO19']
['Walter Scott', 'Sir Walter Scott']
['SCO19']
['Ted Sizer']
['Wannier']
['Walter Fred Bodmer']



In [15]:
cl = 5
dataset2 = DFWrapper(pd.DataFrame.from_records([{'name': name} for name in [''] + clusterer.state.retrieve_cluster_data(sorted(clusterer.state.particles[0], key=lambda c: clusterer.state.clusters[c].size, reverse=True)[cl]) ]))

print('Greedy:')
torch.manual_seed(1)
max_evals = 300; max_particles = 1
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_greedy, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

print('SMC:')
torch.manual_seed(1)
max_evals = 300; max_particles = 10
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_stratified, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

Greedy:


100%|████████████████████████████████| 7/7 [00:02<00:00,  2.57it/s, Particles=1]



Particle 0, weight 1, 4 clusters, 8 points, [3, 2, 2, 1]
['Sir Peter Jackson', 'Sir Peter Robert Jackson', 'Peter Jackson']
['Sir Max Pemberton', 'Max Pemberton']
['Peter Galison', 'Peter Gay']
['Robert Clay Allison']

SMC:


100%|████████████████████████████████| 7/7 [00:05<00:00,  1.19it/s, Particles=3]


Particle 0, weight 0.8, 4 clusters, 8 points, [3, 2, 2, 1]
['Sir Peter Jackson', 'Sir Peter Robert Jackson', 'Peter Jackson']
['Sir Max Pemberton', 'Max Pemberton']
['Peter Galison', 'Peter Gay']
['Robert Clay Allison']

Particle 1, weight 0.1, 3 clusters, 8 points, [5, 2, 1]
['Sir Peter Jackson', 'Sir Peter Robert Jackson', 'Peter Jackson', 'Peter Galison', 'Peter Gay']
['Sir Max Pemberton', 'Max Pemberton']
['Robert Clay Allison']

Particle 2, weight 0.1, 5 clusters, 8 points, [3, 2, 1, 1, 1]
['Sir Peter Jackson', 'Sir Peter Robert Jackson', 'Peter Jackson']
['Sir Max Pemberton', 'Max Pemberton']
['Peter Galison']
['Robert Clay Allison']
['Peter Gay']



In [16]:
cl = 6
dataset2 = DFWrapper(pd.DataFrame.from_records([{'name': name} for name in [''] + clusterer.state.retrieve_cluster_data(sorted(clusterer.state.particles[0], key=lambda c: clusterer.state.clusters[c].size, reverse=True)[cl]) ]))

print('Greedy:')
torch.manual_seed(1)
max_evals = 300; max_particles = 1
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_greedy, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

print('SMC:')
torch.manual_seed(1)
max_evals = 300; max_particles = 10
clusterer2 = SMCClusterer(dataset2, score_fn, max_evals, max_particles, surrogate2, resample_fn=resample_stratified, ClusterClass=BigramCluster)
clusterer2.cluster(rng)
clusterer2.summary(print_cluster_data = True)

Greedy:


100%|████████████████████████████████| 6/6 [00:02<00:00,  2.42it/s, Particles=1]



Particle 0, weight 1, 3 clusters, 7 points, [3, 2, 2]
['Alexander VIII', 'Alexander Lauder', 'Pope Alexander VIII']
['Aleksandr Ostrovsky', 'Alexander Ostrovsky']
['Paul III', 'Pope Paul III']

SMC:


100%|████████████████████████████████| 6/6 [00:04<00:00,  1.23it/s, Particles=9]


Particle 0, weight 0.8, 4 clusters, 7 points, [2, 2, 2, 1]
['Aleksandr Ostrovsky', 'Alexander Ostrovsky']
['Alexander VIII', 'Pope Alexander VIII']
['Paul III', 'Pope Paul III']
['Alexander Lauder']

Particle 1, weight 0.2, 3 clusters, 7 points, [3, 2, 2]
['Alexander VIII', 'Alexander Lauder', 'Pope Alexander VIII']
['Aleksandr Ostrovsky', 'Alexander Ostrovsky']
['Paul III', 'Pope Paul III']

Particle 2, weight 4.7e-06, 5 clusters, 7 points, [2, 2, 1, 1, 1]
['Alexander VIII', 'Pope Alexander VIII']
['Paul III', 'Pope Paul III']
['Alexander Lauder']
['Alexander Ostrovsky']
['Aleksandr Ostrovsky']

Particle 3, weight 1.2e-06, 4 clusters, 7 points, [3, 2, 1, 1]
['Alexander VIII', 'Alexander Lauder', 'Pope Alexander VIII']
['Paul III', 'Pope Paul III']
['Aleksandr Ostrovsky']
['Alexander Ostrovsky']

Particle 4, weight 5.6e-07, 3 clusters, 7 points, [4, 2, 1]
['Alexander VIII', 'Alexander Lauder', 'Pope Alexander VIII', 'Alexander Ostrovsky']
['Paul III', 'Pope Paul III']
['Aleksandr Ostr